In [1]:
import pandas as pd
# pd.set_option('display.max_rows', 500)
# pd.set_option('display.max_colwidth', 500)
import warnings
import time
warnings.filterwarnings("ignore", category=FutureWarning)

import geopandas as gpd
import shapely
import numpy as np
from functools import partial
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

import nomad.io.base as loader
import nomad.stop_detection.utils as utils
import nomad.stop_detection.hdbscan as HDBSCAN
import nomad.stop_detection.lachesis as LACHESIS
import nomad.stop_detection.dbscan as TADBSCAN
import nomad.stop_detection.grid_based as GRID_BASED
import nomad.stop_detection.postprocessing as pp
from nomad.stop_detection.validation import AlgorithmRegistry, compute_stop_detection_metrics

import nomad.visit_attribution.visit_attribution as visits
import nomad.filters as filters
import nomad.city_gen as cg
from nomad.city_gen import City

from nomad.contact_estimation import overlapping_visits, compute_visitation_errors, precision_recall_f1_from_minutes
import pdb

In [2]:
import nomad.city_gen as cg
import nomad.data as data_folder
from pathlib import Path
data_dir = Path(data_folder.__file__).parent
city = City.from_geopackage(data_dir / "garden-city.gpkg")

def plot_metrics_boxplots(df, metrics,
                          algo_order=None, colors=None,
                          figsize=(24, 5.5), save_path=None):
    # --- normalise inputs -------------------------------------------------
    if algo_order is None:
        # preserve appearance order in the DataFrame
        algo_order = df.algorithm.drop_duplicates().tolist()

    if colors is None:
        cmap = plt.colormaps.get_cmap('tab10')
        colors = {a: cmap(i % cmap.N) for i, a in enumerate(algo_order)}
    else:
        # fill in any missing algorithm colour with the next Tab10 entry
        cmap = plt.colormaps.get_cmap('tab10')
        for i, a in enumerate(algo_order):
            colors.setdefault(a, cmap(i % cmap.N))

    # --- figure -----------------------------------------------------------
    fig, axes = plt.subplots(1, len(metrics), figsize=figsize, sharey=False)

    if len(metrics) == 1:          # when only one metric is passed
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        ax.set_facecolor('#EAEAF2')

        # list of series, one per algorithm
        data = [df.loc[df.algorithm == a, metric].dropna() for a in algo_order]

        bp = ax.boxplot(data,
                        positions=range(len(algo_order)),
                        patch_artist=True,
                        widths=0.4,
                        whis=(5, 95),
                        showfliers=False,
                        medianprops={'color': 'black', 'linewidth': 0.6})

        for box, alg in zip(bp['boxes'], algo_order):
            box.set_facecolor(colors[alg])

        ax.grid(axis='y', color='darkgray', linestyle='--', linewidth=0.8)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.set_title(metric.replace('_', ' ').title(), fontsize=16)
        ax.set_xticks([])

    # legend
    handles = [plt.matplotlib.patches.Patch(color=colors[a], label=a)
               for a in algo_order]
    fig.legend(handles, algo_order,
               loc='lower center',
               ncol=len(algo_order),
               bbox_to_anchor=(0.5, -0.015),
               fontsize=15)

    plt.subplots_adjust(bottom=0.1, top=0.95)

    if save_path:
        fig.savefig(f'{save_path}.png', dpi=300)
        fig.savefig(f'{save_path}.svg')

    plt.show()

def classify_building_size_from_id(building_id):
    building = city.buildings_df.loc[building_id]
    n_blocks = len(building.blocks)
    if n_blocks == 1:
        return 'small'
    elif 2 <= n_blocks <= 3:
        return 'medium'
    else:
        return 'big'

def classify_building_type_from_id(building_id):
    building = city.buildings_df.loc[building_id]
    return building.building_type

def classify_dwell(duration):
    if duration < 60:
        return 'low'
    elif 60 <= duration <= 180:
        return 'mid'
    else:
        return 'high'

In [3]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
traj_cols = {'user_id': 'user_id', 'x': 'x', 'y': 'y', 'timestamp': 'timestamp'}

In [4]:
# ── DATA GENERATION ───────────────────────────────────────────────────────────

from nomad.traj_gen import Population

_GEN_N         = 250
_GEN_SEED      = 2025
_GEN_BETA_PING = np.linspace(4, 9, _GEN_N).tolist()
_GEN_HA        = 3 / 4
_GEN_START     = pd.Timestamp("2024-06-01T00:00:00-04:00")
_GEN_END       = pd.Timestamp("2024-06-08T00:00:00-04:00")

_sparse_path  = Path("robustness-of-algorithms/sparse_traj_2")
_diaries_path = Path("robustness-of-algorithms/diaries_2")
_homes_path   = Path("robustness-of-algorithms/homes_2")

if _sparse_path.exists() and _diaries_path.exists():
    print("Data already exists — skipping generation.")
else:
    print("Generating trajectories (EPR model, random home/workplace per agent)...")

    _gen_city = City.from_geopackage(data_dir / "garden-city.gpkg")
    _gen_city._build_hub_network(hub_size=16)
    if _gen_city.grav is None:
        _gen_city.compute_gravity(exponent=2.0, use_proxy_hub_distance=True)
    _gen_city.compute_shortest_paths(callable_only=True)

    _poi_data = pd.DataFrame({
        "building_id": _gen_city.buildings_gdf["id"].values,
        "x": _gen_city.buildings_gdf["door_point"].apply(lambda p: p[0]).values,
        "y": _gen_city.buildings_gdf["door_point"].apply(lambda p: p[1]).values,
    })

    _population = Population(_gen_city)
    _population.generate_agents(N=_GEN_N, seed=_GEN_SEED, name_count=3)

    for _i, _agent in enumerate(tqdm(_population.roster.values(), desc="Generating")):
        _agent.generate_trajectory(
            datetime=_GEN_START,
            end_time=_GEN_END,
            epr_time_res=15,
            dt=1,
            seed=_i,
        )
        _agent.sample_trajectory(
            beta_ping=_GEN_BETA_PING[_i],
            ha=_GEN_HA,
            seed=_i,
            replace_sparse_traj=True,
        )

    _population.reproject_to_mercator(sparse_traj=True, full_traj=False, diaries=True, poi_data=_poi_data)
    _population.save_pop(
        sparse_path=str(_sparse_path),
        diaries_path=str(_diaries_path),
        homes_path=str(_homes_path),
        beta_ping=_GEN_BETA_PING,
        ha=_GEN_HA,
    )
    del _gen_city, _population, _poi_data
    print(f"Done: {_GEN_N} agents → {_diaries_path} / {_sparse_path}")

Generating trajectories (EPR model, random home/workplace per agent)...


Generating: 100%|██████████| 250/250 [01:35<00:00,  2.61it/s]


Done: 250 agents → robustness-of-algorithms/diaries_2 / robustness-of-algorithms/sparse_traj_2


In [5]:
# ── DATA LOADING ──────────────────────────────────────────────────────────────
poi_table = gpd.read_file(data_dir / "garden-city.gpkg")
poi_table = poi_table.rename({'type': 'building_type'}, axis=1)
poi_table['building_size'] = poi_table['id'].apply(classify_building_size_from_id)

diaries_df = loader.from_file("robustness-of-algorithms/diaries_2", format="parquet", **traj_cols)
diaries_df = diaries_df.rename({'location': 'id'}, axis=1)
diaries_df = diaries_df.merge(poi_table[['id', 'building_size', 'building_type']], on='id', how='left')
diaries_df.loc[~diaries_df.id.isna(), 'dwell_length'] = (
    diaries_df.loc[~diaries_df.id.isna(), 'duration'].apply(classify_dwell)
)

sparse_df = loader.from_file("robustness-of-algorithms/sparse_traj_2", format="parquet", **traj_cols)

/Users/carolinechen/Desktop/cs/nomad/.venv/lib/python3.9/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'garden-city.gpkg': 'buildings' (default), 'streets', 'city_properties'. Specify layer parameter to avoid this warning.
  result = read_func(


In [6]:
poi_table

,index,id,building_type,door_cell_x,door_cell_y,size,door_point_x,door_point_y,geometry,building_size
0,p-x12-y11,p-x12-y11,park,13,11,16,13.0,11.5,"POLYGON ((13 9, 13 13, 9 13, 9 9, 13 9))",big
1,h-x7-y8,h-x7-y8,home,8,8,2,8.0,8.5,"POLYGON ((7 7, 8 7, 8 9, 7 9, 7 7))",medium
2,h-x9-y7,h-x9-y7,home,9,8,2,9.5,8.0,"POLYGON ((8 7, 10 7, 10 8, 8 8, 8 7))",medium
3,h-x10-y7,h-x10-y7,home,10,8,1,10.5,8.0,"POLYGON ((11 7, 11 8, 10 8, 10 7, 11 7))",small
4,h-x11-y7,h-x11-y7,home,11,8,1,11.5,8.0,"POLYGON ((12 7, 12 8, 11 8, 11 7, 12 7))",small
...,...,...,...,...,...,...,...,...,...,...
101,r-x2-y7,r-x2-y7,retail,3,7,2,3.0,7.5,"POLYGON ((3 7, 3 8, 1 8, 1 7, 3 7))",medium
102,r-x1-y5,r-x1-y5,retail,0,5,3,1.0,5.5,"POLYGON ((2 4, 2 7, 1 7, 1 4, 2 4))",medium
103,r-x2-y6,r-x2-y6,retail,3,6,1,3.0,6.5,"POLYGON ((3 6, 3 7, 2 7, 2 6, 3 6))",small
104,r-x2-y5,r-x2-y5,retail,3,5,1,3.0,5.5,"POLYGON ((3 5, 3 6, 2 6, 2 5, 3 5))",small


## Analyze completeness

In [7]:
completeness_df = filters.completeness(sparse_df, traj_cols=traj_cols)

In [8]:
plt.figure(figsize=(4,3))
ax = sns.kdeplot(
    data=pd.DataFrame(completeness_df),
    x="0",
    fill=True,
    linewidth=1.5,
    bw_adjust=0.25
)

# cosmetics
ax.set_xlabel("q (Trajectory Completeness)")
ax.set_ylabel("Density")
ax.grid(True, alpha=0.3)
sns.despine(top=True, right=True)

# annotation (top-right corner in axes coords)
ax.text(
    0.99, 0.95,
    f"N = {len(completeness_df)}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9
)

plt.tight_layout()
plt.savefig("q_stat_density.svg", format="svg", bbox_inches="tight")
plt.savefig("q_stat_density.png", format="svg", bbox_inches="tight")
plt.show()

ValueError: Could not interpret value `0` for `x`. An entry with this name does not appear in `data`.

<Figure size 400x300 with 0 Axes>

## Execution for all users

In [ ]:
# ── PREPROCESSING / POST-PROCESSING FUNCTIONS ─────────────────────────────────

def no_op_df(data, **kwargs):
    return data

def no_op_stops(stops, **kwargs):
    return stops

def prejoin_oracle_map(data, diary, **kwargs):
    location = visits.oracle_map(data, diary, timestamp='timestamp', location_id='id')
    return data.join(location)

summarize_stops_with_loc = partial(
    utils.summarize_stop, x='x', y='y',
    keep_col_names=False, passthrough_cols=['id'], complete_output=True,
)

def postjoin_poly_map(data, **kwargs):
    if not isinstance(data, gpd.GeoDataFrame):
        from shapely.geometry import Point
        geometry = [Point(xy) for xy in zip(data['x'], data['y'])]
        data_gdf = gpd.GeoDataFrame(data, geometry=geometry, crs='EPSG:3857')
    else:
        data_gdf = data
        if data_gdf.crs is None:
            data_gdf = data_gdf.set_crs('EPSG:3857', allow_override=True)
    location = visits.point_in_polygon(
        data=data_gdf, poi_table=poi_table, method='majority',
        max_distance=12, cluster_label='cluster', location_id='id',
        x='x', y='y', data_crs='EPSG:3857',
    )
    return data.join(location)

def pad_oracle_stops_long(stops, **kwargs):
    return utils.pad_short_stops(stops, pad=15, dur_min=4, start_timestamp='start_timestamp')


# ── PRE/POST PIPELINE — keyed by family name ──────────────────────────────────

pipeline = {
    'ta-hdbscan':      {'pre': no_op_df,           'post': postjoin_poly_map, 'fix': no_op_stops},
    'oracle':          {'pre': prejoin_oracle_map,  'post': no_op_df,          'fix': pad_oracle_stops_long},
    'tadbscan_coarse': {'pre': no_op_df,            'post': postjoin_poly_map, 'fix': no_op_stops},
    'tadbscan_fine':   {'pre': no_op_df,            'post': postjoin_poly_map, 'fix': no_op_stops},
    'lachesis_coarse': {'pre': no_op_df,            'post': postjoin_poly_map, 'fix': no_op_stops},
    'lachesis_fine':   {'pre': no_op_df,            'post': postjoin_poly_map, 'fix': no_op_stops},
}


# ── ALGORITHM REGISTRY ────────────────────────────────────────────────────────

registry = AlgorithmRegistry()

registry.add_algorithm(HDBSCAN.hdbscan_labels,       family='ta-hdbscan',
                       time_thresh=240, min_pts=3, min_cluster_size=1, include_border_points=True)
registry.add_algorithm(GRID_BASED.grid_based_labels,  family='oracle',
                       time_thresh=600, min_pts=0, location_id='id')
registry.add_algorithm(TADBSCAN.ta_dbscan_labels,     family='tadbscan_coarse',
                       time_thresh=240, min_pts=2, dist_thresh=30)
registry.add_algorithm(TADBSCAN.ta_dbscan_labels,     family='tadbscan_fine',
                       time_thresh=120, min_pts=3, dist_thresh=20)
registry.add_algorithm(LACHESIS.lachesis_labels,      family='lachesis_coarse',
                       dt_max=240, delta_roam=40)
registry.add_algorithm(LACHESIS.lachesis_labels,      family='lachesis_fine',
                       dt_max=120, delta_roam=25)

print(f"Registry: {len(registry)} algorithm configurations")

In [ ]:
# ── METRICS FUNCTION ──────────────────────────────────────────────────────────
# Column names expected by compute_stop_detection_metrics / overlapping_visits.
_VALIDATION_TRAJ_COLS = {
    'user_id':         'user_id',
    'location_id':     'location',
    'start_timestamp': 'start_timestamp',
    'end_timestamp':   'end_timestamp',
    'duration':        'duration',
}

def _prep_stops(stops):
    """Rename stop-table columns to the validation standard."""
    return stops.rename(columns={
        'start_datetime': 'start_timestamp',
        'end_datetime':   'end_timestamp',
        'id':             'location',
    })

def _prep_truth(truth):
    """Rename diary columns to the validation standard and add end_timestamp."""
    t = truth.copy()
    if 'end_timestamp' not in t.columns:
        t['end_timestamp'] = t['timestamp'] + pd.to_timedelta(t['duration'], unit='m')
    return t.rename(columns={'timestamp': 'start_timestamp', 'id': 'location'})

def compute_all_metrics(stops, truth, user, algo):
    """Compute general + per-category metrics using compute_stop_detection_metrics."""
    stops_v = _prep_stops(stops)
    truth_v = _prep_truth(truth)

    # General (all stops)
    gen = compute_stop_detection_metrics(
        stops_v, truth_v.dropna(subset=['location']),
        algorithm=algo, prf_only=False, traj_cols=_VALIDATION_TRAJ_COLS,
    )
    gen.update({'user': user, 'metric_category': 'general', 'category_value': 'all'})
    gen.pop('user_id', None)
    results = [gen]

    # Per-category slices
    for category in ['building_size', 'building_type', 'dwell_length']:
        for val in truth_v[category].dropna().unique():
            truth_sub = truth_v[(truth_v[category] == val) & (truth_v['location'].notna())]
            cat = compute_stop_detection_metrics(
                stops_v, truth_sub,
                algorithm=algo, prf_only=False, traj_cols=_VALIDATION_TRAJ_COLS,
            )
            cat.update({'user': user, 'metric_category': category, 'category_value': val})
            cat.pop('user_id', None)
            results.append(cat)

    return results

### OPTIMIZED LOOP

In [ ]:
results_list = []

for user in tqdm(diaries_df.user_id.unique(), desc='Processing users'):
    sparse = sparse_df[sparse_df['user_id'] == user].copy()
    truth  = diaries_df[diaries_df['user_id'] == user].copy()

    for algo in registry:
        name = algo["family"]
        pipe = pipeline[name]

        # PRE
        processed_sparse = pipe["pre"](data=sparse, diary=truth)

        # ALGORITHM
        labels = registry.time_call(algo, processed_sparse, traj_cols=traj_cols)

        # POST
        data_with_clusters  = processed_sparse.join(labels)
        data_with_locations = pipe["post"](data=data_with_clusters)
        stops = data_with_locations[data_with_locations.cluster != -1].groupby(
            'cluster', as_index=False
        ).apply(summarize_stops_with_loc, include_groups=False)
        stops = pipe["fix"](stops=stops, data_with_clusters=data_with_locations, params=algo["params"])

        # METRICS
        metrics = compute_all_metrics(stops, truth, user, name)
        if metrics:
            metrics[0]['execution_time'] = registry._timings[-1]['elapsed_s']
        results_list.extend(metrics)

results_df = pd.DataFrame(results_list)
print("Processing Complete!")

In [ ]:
try:
    results_df = pd.read_csv("results.csv")
except:
    results_df.to_csv('results.csv', index=False)

In [ ]:
general_metrics_df = results_df[results_df['metric_category'] == 'general'].copy()

In [ ]:
# BOOTSTRAPPING GROUPBY
agg_keys = ['missed_fraction','merged_fraction','split_fraction','precision','recall','f1']
ua = general_metrics_df.groupby(['user','algorithm'], as_index=False)[agg_keys].median()
users = ua['user'].unique()
output = []
for _ in range(2000):
    draw = np.random.choice(users, size=len(users), replace=True)
    bs = ua[ua.user.isin(draw)]
    output.append(bs.groupby('algorithm', as_index=False)[agg_keys].median())
metrics_bootstrap_df = pd.concat(output, ignore_index=True)

In [ ]:
# # For debugging!!!
# metrics = ['missed_fraction', 'merged_fraction', 'split_fraction', 'precision', 'recall', 'f1']
# plot_metrics_boxplots(metrics_bootstrap_df, metrics, algo_order=None, colors=None, save_path=None)

### Plot boostrapped per-stop metrics for general trajectory

In [ ]:
# base colors for coarse versions
shade = lambda c,f=0.6: mcolors.to_hex(tuple(f*x for x in mcolors.to_rgb(c)))
raw = {'oracle':'royalblue','ta-hdbscan':'darkorange','lachesis_coarse':'palevioletred','tadbscan_coarse':'limegreen'}

_base = {k:mcolors.to_hex(mcolors.to_rgb(v)) for k,v in raw.items()}

algo_order = ['oracle','ta-hdbscan','lachesis_coarse','tadbscan_coarse','lachesis_fine', 'tadbscan_fine']
colors = {a:(_base[a] if a in _base else shade(_base[a.split('_')[0]+'_coarse'])) for a in algo_order}

In [ ]:
metrics = ['missed_fraction', 'merged_fraction', 'split_fraction', 'precision', 'recall', 'f1']
plot_metrics_boxplots(metrics_bootstrap_df, metrics, algo_order=algo_order, colors=colors, save_path='errors_per_stop')

In [ ]:
# ylabels = {m:f"% {m.split('_')[0]}" for m in ['precision','recall','f1']}
# metrics = ['precision', 'recall', 'f1']
# plot_metrics_boxplots( metrics_bootstrap_df, metrics, algo_order=None, colors=None, save_path='errors_per_stop')

In [ ]:
### Plot pr, rec, f1 per each group in 4 subplots. 

### Plot of F1 scores for each q

In [ ]:
def plot_f1_vs_q(general_df, completeness_df, save_path=None):
    """Three-panel plot: oracle vs hdbscan / tadbscan_coarse / lachesis_coarse."""

    # --- palette & comparison pairs --------------------------------------
    pairs  = [('oracle', 'ta-hdbscan'),
              ('oracle', 'tadbscan_coarse'),
              ('oracle', 'lachesis_coarse')]

    colors = {'oracle': 'royalblue',
              'ta-hdbscan': 'darkorange',
              'tadbscan_coarse': 'limegreen',
              'lachesis_coarse': 'palevioletred'}

    # --- merge: just rename q_stat -> q ----------------------------------
    comp = completeness_df.rename(columns={'q_stat': 'q'})[['user_id', 'q']]
    df   = general_df.merge(comp, left_on='user', right_on='user_id', how='left')

    # --- figure ----------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(13, 2.75), sharey=True)

    for ax, (a1, a2) in zip(axes, pairs):
        ax.set_facecolor('#EAEAF2')            # seaborn-like bg

        for alg in (a1, a2):
            sub = df[df.algorithm == alg].dropna(subset=['q']).sort_values('q')
            if sub.empty:
                continue
            sm  = lowess(sub['f1'], sub['q'], frac=0.6)
            ax.plot(sm[:, 0], sm[:, 1], color=colors[alg], lw=2)
            ax.scatter(sub['q'], sub['f1'], color=colors[alg], s=6, alpha=0.25)

        ax.grid(axis='y', color='darkgray', linestyle='--', lw=0.8)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.set_xlabel("q (trajectory completeness)")
        ax.set_xticks(np.linspace(0, 1, 6))
        ax.tick_params(axis='both', labelsize=10)

    axes[0].set_ylabel("F1 score")

    # --- single legend ---------------------------------------------------
    handles = [plt.matplotlib.patches.Patch(color=colors[a], label=a) for a in colors]
    fig.legend(handles, colors, loc='lower center',
               ncol=len(colors), bbox_to_anchor=(0.5, -0.12))

    plt.tight_layout()

    if save_path:
        fig.savefig(f"{save_path}.png", dpi=300, bbox_inches='tight')
        fig.savefig(f"{save_path}.svg", bbox_inches='tight')

    plt.show()

plot_f1_vs_q(general_metrics_df, completeness_df,
             save_path="f1_vs_q")

### Metrics for each category value

In [ ]:
# for each user, algo, and category_value
if 'f1_as_pct_orac' not in results_df:
    oracle_df = results_df.loc[results_df.algorithm == 'oracle', ['user', 'category_value','f1']].rename(columns={'f1':'f1_oracle'})
    results_df = results_df.merge(oracle_df, on=['user','category_value'],  how='left')
    results_df['f1_as_pct_orac'] = 100 * results_df['f1'] / results_df['f1_oracle']

In [ ]:
#table_results = results_df.loc[results_df.algorithm.isin(['ta-hdbscan', 'lachesis_coarse','tadbscan_coarse', 'lachesis_fine', 'tadbscan_fine'])]#
table_results = results_df.loc[results_df.algorithm.isin(['ta-hdbscan', 'lachesis_coarse','tadbscan_coarse'])]
table_results = table_results.loc[~table_results.category_value.isin(['all', 'big', 'medium', 'small', 'mid'])]
table_results = table_results.groupby(['metric_category', 'category_value', 'algorithm'], as_index=True)[['f1', 'f1_as_pct_orac']].median().round(2)
table_results.to_csv('results_by_category.csv', index=True)
table_results

# Updated Plots (Post-Netmob)

In [ ]:
print(stops.columns.tolist())

In [ ]:
poi_table = poi_table.set_crs('EPSG:3857', allow_override=True)

In [ ]:
results_list = []

for user in tqdm(diaries_df.user_id.unique(), desc='Processing users'):
    sparse = sparse_df[sparse_df['user_id'] == user].copy()
    truth  = diaries_df[diaries_df['user_id'] == user].copy()

    for algo in registry:
        name = algo["family"]
        pipe = pipeline[name]

        # PRE
        processed_sparse = pipe["pre"](data=sparse, diary=truth)

        # ALGORITHM
        labels = registry.time_call(algo, processed_sparse, traj_cols=traj_cols)

        # POST
        data_with_clusters  = processed_sparse.join(labels)
        data_with_locations = pipe["post"](data=data_with_clusters)
        stops = data_with_locations[data_with_locations.cluster != -1].groupby(
            'cluster', as_index=False
        ).apply(summarize_stops_with_loc, include_groups=False)
        stops = pipe["fix"](stops=stops, data_with_clusters=data_with_locations, params=algo["params"])

        # METRICS
        metrics = compute_all_metrics(stops, truth, user, name)
        if metrics:
            metrics[0]['execution_time'] = registry._timings[-1]['elapsed_s']
        results_list.extend(metrics)

results_df = pd.DataFrame(results_list)
print("Processing Complete!")